In [14]:
import requests
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import json

In [ ]:
# Seattle lat, long
lat, lon = 47.6062, -122.3321

headers = {
    'User-Agent': 'WildfireOptimizationProject - StudentResearch'
}

try:
    # Step 1: Metadata
    metadata_response = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=headers, timeout=10).json()
    forecast_url = metadata_response['properties']['forecastHourly']

    # Step 2: Forecast Data
    weather_data = requests.get(forecast_url, headers=headers, timeout=10).json()
    current_forecast = weather_data['properties']['periods'][0]
    
    print("Success! Data loaded into memory.")

except requests.exceptions.Timeout:
    print("The NWS API timed out. They are currently experiencing heavy server load.")

Fetching hourly forecast...
Success! Data loaded into memory.


In [12]:
# Extract the critical threat vectors
wind_speed_str = current_forecast['windSpeed'] # Gets '15 mph'
wind_speed = int(''.join(filter(str.isdigit, wind_speed_str))) # Converts to 15
wind_direction = current_forecast['windDirection']
temp = current_forecast['temperature']
rh = current_forecast['relativeHumidity']['value']
precip = current_forecast['probabilityOfPrecipitation']['value'] or 0

print(f"Active Threat Vector:")
print(f"Temp: {temp}°F | Wind: {wind_speed} mph from {wind_direction}")
print(f"RH: {rh}% | Precipitation Chance: {precip}%")

Active Threat Vector:
Temp: 57°F | Wind: 6 mph from SSW
RH: 82% | Precipitation Chance: 1%


In [19]:
# 1. Load the foundation you built in the first notebook
assets_gdf = gpd.read_file("data/seattle_assets.geojson")

# 2. Get the extreme outer boundaries of your assets
xmin, ymin, xmax, ymax = assets_gdf.total_bounds

# 3. Define the grid resolution 
# 0.005 degrees of latitude/longitude is roughly a 500m x 500m square
grid_size = 0.004

# 4. Generate the matrix of squares with indices
cols = list(np.arange(xmin, xmax, grid_size))
rows = list(np.arange(ymin, ymax, grid_size))

polygons = []
grid_nodes = [] # <--- THIS IS THE MISSING VARIABLE

for c_idx, x in enumerate(cols[:-1]):
    for r_idx, y in enumerate(rows[:-1]):
        # 1. Create the shape
        polygons.append(Polygon([(x,y), (x+grid_size, y), (x+grid_size, y+grid_size), (x, y+grid_size)]))
        
        # 2. Store the index tuple so 'grid_nodes' is defined
        grid_nodes.append((c_idx, r_idx))

# Now create the GDF
grid_gdf = gpd.GeoDataFrame({'geometry': polygons}, crs=assets_gdf.crs)

# Now this line will work because 'grid_nodes' exists!
grid_gdf['node_id'] = [f"{c}_{r}" for c, r in grid_nodes] 
grid_gdf.to_file("data/spatial_backbone.geojson", driver='GeoJSON')

In [13]:
# 1. Define the Red Flag Risk Formula 

# Temperature (>80°F is critical)
# If over 80F, it's a severe threat (1.0+). If under, it scales down.
temp_factor = 1.2 if temp > 80 else max(0.2, temp / 80)

# Wind Speed (>20 mph is critical)
# High winds act as an exponential multiplier.
wind_factor = 1.5 if wind_speed > 20 else max(1.0, 1 + (wind_speed / 20))

# Relative Humidity (<30% is critical)
# If RH is below 30%, it offers ZERO dampening effect (multiplier is 1.0 or higher).
# If RH is high, it heavily dampens the fire spread.
moisture_factor = 1.0 if rh < 30 else ((100 - rh) / 70)

# Precipitation (Rain kills fires)
rain_dampener = (100 - precip) / 100

# 2. Calculate the final Red Flag Index
base_threat = temp_factor * wind_factor * moisture_factor * rain_dampener

# Cap the maximum risk at 1.0 (100%) to keep the optimization math stable
base_threat = min(1.0, max(0.01, base_threat))

grid_gdf['base_spread_risk'] = base_threat
print(f"Red Flag Environmental Risk calculated at: {base_threat:.2%}")

Red Flag Environmental Risk calculated at: 23.58%


In [15]:
# Bundle the live variables into a dictionary
weather_export = {
    "wind_speed": wind_speed,
    "wind_direction": wind_direction,
    "temp": temp,
    "rh": rh,
    "precip": precip
}

# Save it to the data folder
with open('data/live_weather_config.json', 'w') as f:
    json.dump(weather_export, f, indent=4)